# Lesson 10 — Text to numbers: n-grams and the hashing trick

Lesson 10 — Text to numbers: n-grams and the hashing trick
==========================================================

GURU NOTE
---------
The logistic model ate 32 pre-computed features. The payload MLP is different: it
eats the RAW decoded text of the query and body. But a neural net needs a
fixed-length vector of numbers, and text is variable-length characters. How do we
turn "' OR 1=1--" into exactly 512 numbers, cheaply, and identically in Rust?

Three ideas, and they're all in neural.featurize:

  1. CHARACTER N-GRAMS. Slide a window over the text and collect every substring
     of length 3, 4, 5. "' or 1=1" -> "' o", "' or", "' or ", "or ", ... These
     little fragments are the vocabulary of attacks: "<scr", "=1=1", "un io" etc.

  2. THE HASHING TRICK. There are millions of possible n-grams. Instead of a
     dictionary, hash each one and use (hash mod 512) as its slot in a fixed
     512-length vector, adding 1 each time. No vocabulary file, fixed size, and
     trivially reproducible in Rust. The cost: sometimes two different n-grams
     land in the same slot (a 'collision'). We accept that noise.

  3. TIDY UP. log1p (so a fragment repeated 100x doesn't dominate) then
     L2-normalize (so a 2KB body and a 30-char query are on the same scale).

This lesson builds all three, exactly as neural.py does — including the same
FNV-1a hash with the same constants, so your vectors would match the engine's.

Run me:  python lessons/lesson_10_text_hashing_ngrams.py

## Setup

These notebooks *import* the real code from `lessons/*.py` — one source of truth, no forked copies. This cell finds that folder.

In [1]:
import sys, pathlib
for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (base / 'lessons' / '_util.py').exists():
        sys.path.insert(0, str(base / 'lessons'))
        print('lessons on path:', base / 'lessons'); break
else:
    raise RuntimeError('Open this notebook inside the xorguard-ml-course folder.')

lessons on path: /Users/anil/Documents/developer/vscodeworkspaces/xordev/xorguard-ml-course/lessons


## The key code (shown live from the source file)

Read it, then copy any function into a new cell and tweak it — the originals live in `lessons/lesson_10_text_hashing_ngrams.py`.

In [2]:
import inspect
from lesson_10_text_hashing_ngrams import fnv1a_32, featurize
for fn in (fnv1a_32, featurize):
    print(inspect.getsource(fn)); print('-' * 72)

def fnv1a_32(data: bytes) -> int:
    """A tiny, fast, well-mixing hash. Chosen because it's trivial to reproduce
    in Rust byte-for-byte. XOR each byte in, multiply by a prime, keep 32 bits."""
    h = FNV_OFFSET
    for b in data:
        h ^= b
        h = (h * FNV_PRIME) & 0xFFFFFFFF
    return h

------------------------------------------------------------------------
def featurize(text: str, dim: int = HASH_DIM) -> np.ndarray:
    """Text -> fixed 512-vector. IDENTICAL logic to neural.featurize.

    1) count 3,4,5-gram occurrences into `dim` hashed buckets
    2) log1p to squash repetition
    3) L2-normalize to unit length
    """
    raw = text.encode("utf-8", "replace")[:MAX_CHARS]
    v = np.zeros(dim)
    for n in NGRAMS:
        for i in range(len(raw) - n + 1):
            bucket = fnv1a_32(raw[i:i + n]) % dim
            v[bucket] += 1.0
    v = np.log1p(v)                     # 100 hits -> ~4.6, not 100
    nrm = np.linalg.norm(v)
    if nrm > 0:
        v /= nrm     

## Run the full narrated lesson

This executes the same walkthrough as `python lessons/lesson_10_text_hashing_ngrams.py`, printing each step with its result.

In [3]:
from lesson_10_text_hashing_ngrams import main
main()


══════════════════════════════════════════════════════════
Lesson 10 — Text to numbers: n-grams and the hashing trick
══════════════════════════════════════════════════════════

▸ Step 1 — character n-grams. The payload's 'words' are tiny fragments.
    3-grams of "' or 1=1": ["' o", ' or', 'or ', 'r 1', ' 1=', '1=1'] ...  (6 total)
    4-grams of "' or 1=1": ["' or", ' or ', 'or 1', 'r 1=', ' 1=1'] ...  (5 total)
    Fragments like "1=1" and "' o" are exactly the tells of injection.

▸ Step 2 — the hash. Each fragment -> a bucket number in [0, 512).
    fnv1a(b'<scr') =  937811753  ->  bucket 297
    fnv1a(b'1=1') = 1679292774  ->  bucket 358
    fnv1a(b'uni') = 1339334217  ->  bucket 73
    fnv1a(b'sel') = 3190225515  ->  bucket 107
    No dictionary needed. The hash IS the lookup. Reproducible anywhere.

▸ The cost of hashing: COLLISIONS. Different fragments, same bucket.
    'abk' and 'ala' BOTH hash to bucket 115.
    With 512 buckets and thousands of fragments, collisions are co

## Exercises

Try these in new cells below. Worked answers are in `solutions/lesson_10_solution.py`.

```
1. Raise HASH_DIM to 4096 and re-run the collision search. Do collisions get
   rarer? Featurize the SQLi payload at dim=512 vs dim=4096 and compare the
   non-zero counts. This is the size/accuracy trade the repo made.

2. Featurize "' OR 1=1--" and its URL-encoded form "%27%20OR%201=1--" WITHOUT
   decoding. How similar are they? Now lowercase+urldecode both first (like
   neural.payload_text) and compare again. Why does decoding matter so much?

3. Compute the cosine similarity between an XSS payload and the SAME payload
   with the tag uppercased ("<SCRIPT>"). Then lowercase both first. This
   previews Lesson 12: case-swapping is a real evasion the repo defends against.
```

In [4]:
# your work here
